In [1]:
import pandas as pd
import dask.dataframe as dd 
import numpy as np
import datetime as dt
from datetime import date
pd.set_option('display.max_columns', None)

In [2]:
### Facility CBSA codes 
### Date: 021125 
### Author: Nadia Ghazali 
### Description: The purpose of this script is to create a crosswalk of area wage indexes for each nursing facility in 2019 to use with the 2019 TAF long term care claims. 
### To do so, I used a series of files to crosswalk between facility NPI (the identifier in the TAF claims) and CBSA (Core Based Statistical Area). 
### Urban area index is assigned based on CBSA, while rural area wage index is the same for all counties within a given state. 

### I use the following files to crosswalk between facility NPI and CBSA code:
### NPI XWALK: facility NPI -> zip code 
### PROVIDER INFO: zip code -> county SSA code 
### CBSA FIPS XW: county SSA code -> CBSA code
### URBAN WAGE INDEX: CBSA code -> area wage index 



In [2]:


def prep_area_wage_file(year, filetype, state_list): 
    print(f'***YEAR: {year}')



    all_states_claims_npis = list_all_states_npis(year, filetype, state_list)
    
    claims_npi = merge_claims_npi(all_states_claims_npis, npi_xwalk)
    claims_npi_provider = merge_provider_info(claims_npi)
    claims_npi_zip_county_cbsa_merge = merge_facility_cbsa_codes(claims_npi_provider)
    area_wage_index = merge_wage_indexes(claims_npi_zip_county_cbsa_merge, year)
    area_wage_index.to_csv(f'/gpfs/data/cms-share/duas/56930/Nadia/python_code/medicaid_pmt_rates/wage_index/{year}_facility_area_wage_index.csv')
    
    # print(f'{year} area_wage_index read out') 

    print(len(area_wage_index['NPI'].value_counts()))
    return area_wage_index 




In [3]:
def list_all_states_npis(year, filetype, state_list):

    state_dataframes = []

    if filetype == 'max':

        npi_col = ['NPI']
    else: 
        npi_col = ['facility_npi']

    for state in state_list: 
            claims = pd.read_parquet(f'/gpfs/data/cms-share/duas/56930/Nadia/data/medicaid_NF_pmt_rates/{filetype}_lt_NF_claims/{year}/{state}', columns=npi_col)
            claims['STATE'] = f'{state}'
            
            # drop duplicates
            claims = claims.drop_duplicates(subset=npi_col, keep='first').reset_index(drop=True)

            state_dataframes.append(claims)
        
    
    all_states_claims_npis = pd.concat(state_dataframes)

    if npi_col == ['facility_npi']:
        all_states_claims_npis = all_states_claims_npis.rename(columns={'facility_npi':'NPI'})

    return all_states_claims_npis


In [ ]:
def clean_npi_xwalk(): 
    # read in NPI crosswalk 
    npi_xwalk = pd.read_csv('/gpfs/data/cms-share/duas/56930/Joe/bed_blocking/NPI/npidata.csv',header=0, 
                            usecols=['NPI','Entity Type Code', 'Provider Business Practice Location Address Postal Code'], 
                            dtype={'NPI':'string','Entity Type Code':'string','Provider Business Practice Location Address Postal Code': 'string'})
    print(npi_xwalk.columns)
    
    npi_xwalk = npi_xwalk.rename(columns={'Entity Type Code':'Entity_Type_Code', 'Provider Business Practice Location Address Postal Code':'Practice_Zip'})
                                          
    # keep 'organization' and drop 'individual' entity types 
    npi_xwalk = npi_xwalk.loc[npi_xwalk['Entity_Type_Code']=='2']
    
    # format zip codes 
    npi_xwalk['Practice_Zip'] = npi_xwalk['Practice_Zip'].astype(str).str[:5]

    return npi_xwalk

npi_xwalk = clean_npi_xwalk()

In [5]:
def merge_claims_npi(all_states_claims_npis, npi_xwalk): 
    # merge MAX/TAF claims with NPI crosswalk 
    claims_npi = all_states_claims_npis.merge(npi_xwalk, how='left', on='NPI', indicator='claims_npi_merge')
    claims_npi = claims_npi.loc[claims_npi['claims_npi_merge']=='both']
    
    print(claims_npi['claims_npi_merge'].value_counts())

    return claims_npi
    

In [6]:
def merge_provider_info(claims_npi): 

    provider_info = pd.read_csv('/gpfs/data/cms-share/duas/56930/Nadia/data/SNF_quality/NH_ratings/nh_archive_12_2019/ProviderInfo_Download.csv', 
                            usecols=['ZIP','STATE','COUNTY_SSA','PARTICIPATION_DATE','County_name'], header=0, dtype='str')

   
    provider_info = provider_info.drop_duplicates(subset=['ZIP','COUNTY_SSA'], keep='first')
    
    print(provider_info.columns.to_list())
    
    state_codes = pd.read_csv('/gpfs/data/cms-share/duas/56930/Nadia/python_code/medicaid_pmt_rates/state_SSA_codes.csv', dtype={'STATE_CD':'str', 'STATE_NAME':'str', 'STATE':'str'})
    
    provider_info = provider_info.merge(state_codes, how='left', on='STATE', indicator='state_code_merge')
    provider_info = provider_info.drop(columns=['STATE'])
    
    provider_info = provider_info.loc[provider_info['state_code_merge']=='both']
    provider_info = provider_info.drop(columns=['state_code_merge'])

    claims_npi_provider = claims_npi.merge(provider_info, how='left', left_on='Practice_Zip', right_on='ZIP', indicator='claims_provider_zip_merge')
    claims_npi_provider = claims_npi_provider.loc[claims_npi_provider['claims_provider_zip_merge']=='both']
    print(claims_npi_provider['claims_provider_zip_merge'].value_counts())
    return claims_npi_provider


In [7]:
def merge_facility_cbsa_codes(claims_npi_provider): 
    county_ssa_cbsa = pd.read_csv('/gpfs/data/cms-share/duas/56930/Nadia/python_code/medicaid_pmt_rates/ssa_fips_state_county2019.csv', header=0, dtype={'county':'str', 'state':'str', 'ssacd':'str', 'fipscounty':'str', 'cbsa':'str', 'cbsaname':'str'})
    county_ssa_cbsa['ssacd'] = county_ssa_cbsa['ssacd'].str[-3:]
    claims_npi_zip_county_cbsa_merge = claims_npi_provider.merge(county_ssa_cbsa, how='left', left_on=['COUNTY_SSA','STATE'], right_on=['ssacd','state'], indicator='county_ssa_merge')
    print(claims_npi_zip_county_cbsa_merge['county_ssa_merge'].value_counts())

    return claims_npi_zip_county_cbsa_merge

In [8]:
def merge_wage_indexes(claims_npi_zip_county_cbsa_merge, year): 
    urban_wage_index = pd.read_csv(f'/gpfs/data/cms-share/duas/56930/Nadia/python_code/medicaid_pmt_rates/wage_index/urban_cbsa_wage_index_{year}.csv', 
                                   dtype = 'str')
    urban_wage_index = urban_wage_index.rename(columns={'CBSA Code':'CBSA_Code','Wage Index':'Wage_Index'})

    cbsa_wage_index_merge = claims_npi_zip_county_cbsa_merge.merge(urban_wage_index, how='left', left_on='cbsa', right_on='CBSA_Code', indicator='urban_CBSA_merge')
    urban_cbsa_wage_index_merge = cbsa_wage_index_merge.loc[cbsa_wage_index_merge['urban_CBSA_merge']=='both']
    print('first merge: ' + str(cbsa_wage_index_merge['urban_CBSA_merge'].value_counts()))
    
    urban_cbsa_wage_index_merge['urban/rural'] = 'urban'

    rural_cbsa_wage_index_merge = cbsa_wage_index_merge.loc[cbsa_wage_index_merge['urban_CBSA_merge']=='left_only']
    rural_cbsa_wage_index_merge['urban/rural'] = 'rural'
    rural_cbsa_wage_index_merge = rural_cbsa_wage_index_merge.drop(columns=['Wage_Index'])
    print('second_merge: ' + str(rural_cbsa_wage_index_merge['urban_CBSA_merge'].value_counts()))
    
    # for failed merges, try to merge with rural area wage index file (merge based on state only) 
    rural_wage_index = pd.read_csv(f'/gpfs/data/cms-share/duas/56930/Nadia/python_code/medicaid_pmt_rates/wage_index/rural_wage_index_{year}.csv',
                                   usecols = ['Nonurban Area', 'STATE', 'Wage Index'],
                                   dtype='str')
    rural_wage_index = rural_wage_index.rename(columns={'Wage Index':'Wage_Index'})
    rural_cbsa_wage_index_merge = rural_cbsa_wage_index_merge.merge(rural_wage_index, how='left', left_on='STATE', right_on='STATE')
    facility_area_wage_index = pd.concat([urban_cbsa_wage_index_merge, rural_cbsa_wage_index_merge])
    facility_area_wage_index = facility_area_wage_index.sort_values(by=['STATE'])

    
    facility_area_wage_index['Wage_Index'] = facility_area_wage_index['Wage_Index'].replace('-----', np.nan)
    facility_area_wage_index['Wage_Index'] = facility_area_wage_index['Wage_Index'].astype(float)
    facility_area_wage_index['facility_npi'] = facility_area_wage_index['NPI'].astype('str')

    return facility_area_wage_index


In [ ]:
# Read in 2019 cleaned claims 

state_dataframes = []

## test with Wyoming 
year = '2019' 
taf_state_list = ['AK', 'AL', 'AR', 'AZ', 'CA', 'CO', 'CT', 'DC', 'DE', 
                              'FL', 'GA', 'HI', 'IA', 'ID', 'IL', 'IN', 'KS', 'KY',
                              'LA', 'MA', 'MD', 'ME', 'MI', 'MO', 'MS', 'MT', 'NC',
                              'ND', 'NE', 'NH', 'NJ', 'NM', 'NV', 'NY', 'OH', 'OK', 
                              'OR', 'PA', 'RI', 'SC', 'SD', 'TN', 'TX', 'UT', 'VA', 
                              'VT', 'WA', 'WI', 'WV', 'WY']

# for state in taf_state_list:  

for state in taf_state_list: 
    taf_claims = pd.read_parquet(f'/gpfs/data/cms-share/duas/56930/Nadia/data/medicaid_NF_pmt_rates/taf_lt_NF_claims/{year}/{state}', columns=['facility_npi'])
    taf_claims['STATE'] = f'{state}'

    # drop duplicates
    taf_claims = taf_claims.drop_duplicates(subset=['facility_npi'], keep='first')
    

    
    state_dataframes.append(taf_claims)


    
all_states_claims_npis = pd.concat(state_dataframes)


In [54]:
# read in NPI crosswalk 
npi_xwalk = pd.read_csv('/gpfs/data/cms-share/duas/56930/Joe/bed_blocking/NPI/npidata.csv',header=0, usecols=['NPI','Entity Type Code', 'Provider Business Practice Location Address Postal Code'], 
                        dtype={'NPI':'string','Entity Type Code':'string','Provider Business Practice Location Address Postal Code': 'string'})
print(npi_xwalk.columns)

npi_xwalk = npi_xwalk.rename(columns={'Entity Type Code':'Entity_Type_Code', 'Provider Business Practice Location Address Postal Code':'Practice_Zip'})
                                      
# keep 'organization' and drop 'individual' entity types 
npi_xwalk = npi_xwalk.loc[npi_xwalk['Entity_Type_Code']=='2']

# format zip codes 
npi_xwalk['Practice_Zip'] = npi_xwalk['Practice_Zip'].astype(str).str[:5]

Index(['NPI', 'Entity Type Code',
       'Provider Business Practice Location Address Postal Code'],
      dtype='object')


In [8]:
print(npi_xwalk.head(10))

           NPI Entity_Type_Code Practice_Zip
2   1497758544                2        28304
5   1023011178                2        94559
15  1023011079                2        60450
21  1487657433                2        61554
27  1740283795                2        50036
34  1982607933                2        21502
36  1700889755                2        21804
38  1427051473                2        49431
39  1336142389                2        13413
41  1962405910                2        35205


In [55]:
# merge TAF claims with NPI crosswalk 
claims_npi = all_states_claims_npis.merge(npi_xwalk, how='left', left_on='facility_npi', right_on='NPI', indicator='claims_npi_merge')
claims_npi = claims_npi.loc[claims_npi['claims_npi_merge']=='both']

print(claims_npi['claims_npi_merge'].value_counts())



claims_npi_merge
both          21020
left_only         0
right_only        0
Name: count, dtype: int64


In [10]:
provider_info = pd.read_csv('/gpfs/data/cms-share/duas/56930/Nadia/python_code/medicaid_pmt_rates/ProviderInfo_Download.csv', usecols=['ZIP','STATE','COUNTY_SSA','County_name'], header=0, dtype={'ZIP':'str','COUNTY_SSA':'str','County_name':'str'})
# provider_info = pd.read_csv('/gpfs/data/cms-share/duas/56930/Nadia/python_code/medicaid_pmt_rates/ProviderInfo_Download.csv')

provider_info = provider_info.drop_duplicates(subset=['ZIP','COUNTY_SSA'], keep='first')

print(provider_info.columns.to_list())

state_codes = pd.read_csv('/gpfs/data/cms-share/duas/56930/Nadia/python_code/medicaid_pmt_rates/state_SSA_codes.csv', dtype={'STATE_CD':'str', 'STATE_NAME':'str', 'STATE':'str'})

provider_info = provider_info.merge(state_codes, how='left', on='STATE', indicator='state_code_merge')
provider_info = provider_info.drop(columns=['STATE'])

provider_info = provider_info.loc[provider_info['state_code_merge']=='both']
provider_info = provider_info.drop(columns=['state_code_merge'])

['STATE', 'ZIP', 'COUNTY_SSA', 'County_name']


In [11]:
claims_npi_provider = claims_npi.merge(provider_info, how='left', left_on='Practice_Zip', right_on='ZIP', indicator='taf_provider_zip_merge')
claims_npi_provider = claims_npi_provider.loc[claims_npi_provider['taf_provider_zip_merge']=='both']

In [14]:
print(len(claims_npi_provider['facility_npi'].value_counts()))

19013


In [15]:
county_ssa_cbsa = pd.read_csv('/gpfs/data/cms-share/duas/56930/Nadia/python_code/medicaid_pmt_rates/ssa_fips_state_county2019.csv', header=0, dtype={'county':'str', 'state':'str', 'ssacd':'str', 'fipscounty':'str', 'cbsa':'str', 'cbsaname':'str'})
county_ssa_cbsa['ssacd'] = county_ssa_cbsa['ssacd'].str[-3:]


In [16]:
# merge facility CBSA codes 

claims_npi_zip_county_cbsa_merge = claims_npi_provider.merge(county_ssa_cbsa, how='left', left_on=['COUNTY_SSA','STATE'], right_on=['ssacd','state'], indicator='county_ssa_merge')

In [17]:
urban_wage_index = pd.read_csv('/gpfs/data/cms-share/duas/56930/Nadia/python_code/medicaid_pmt_rates/wage_index/urban_cbsa_wage_index_2019.csv')
urban_wage_index = urban_wage_index.rename(columns={'CBSA Code':'CBSA_Code','Wage Index':'Wage_Index'})
# urban_wage_index['CBSA_Code'] = urban_wage_index['CBSA_Code'].astype(str).str.zfill(5)


In [18]:
cbsa_wage_index_merge = claims_npi_zip_county_cbsa_merge.merge(urban_wage_index, how='left', left_on='cbsa', right_on='CBSA_Code', indicator='urban_CBSA_merge')


In [19]:
urban_cbsa_wage_index_merge = cbsa_wage_index_merge.loc[cbsa_wage_index_merge['urban_CBSA_merge']=='both']
urban_cbsa_wage_index_merge['urban/rural'] = 'urban'

/tmp/ipykernel_1859135/3704266095.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  urban_cbsa_wage_index_merge['urban/rural'] = 'urban'


In [21]:
print(urban_cbsa_wage_index_merge['urban_CBSA_merge'].value_counts())

urban_CBSA_merge
both          13949
left_only         0
right_only        0
Name: count, dtype: int64


In [23]:
rural_cbsa_wage_index_merge = cbsa_wage_index_merge.loc[cbsa_wage_index_merge['urban_CBSA_merge']=='left_only']
rural_cbsa_wage_index_merge['urban/rural'] = 'rural'
rural_cbsa_wage_index_merge = rural_cbsa_wage_index_merge.drop(columns=['Wage_Index'])
print(rural_cbsa_wage_index_merge['urban_CBSA_merge'].value_counts())


urban_CBSA_merge
left_only     5463
right_only       0
both             0
Name: count, dtype: int64


/tmp/ipykernel_1859135/848153810.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rural_cbsa_wage_index_merge['urban/rural'] = 'rural'


In [26]:
# for failed merges, try to merge with rural area wage index file (merge based on state only) s
rural_wage_index = pd.read_csv('/gpfs/data/cms-share/duas/56930/Nadia/python_code/medicaid_pmt_rates/wage_index/2019_rural_wage_index.csv', dtype={'State Code':'str'})
rural_wage_index = rural_wage_index.rename(columns={'Wage Index':'Wage_Index'})

In [27]:
rural_cbsa_wage_index_merge = rural_cbsa_wage_index_merge.merge(rural_wage_index, how='left', left_on='STATE', right_on='STATE')

In [28]:
print(rural_cbsa_wage_index_merge.shape[0])
print(rural_cbsa_wage_index_merge.head(10))

5463
  facility_npi STATE         NPI Entity_Type_Code Practice_Zip taf_npi_merge  \
0   1730548462    AK  1730548462                2        99835          both   
1   1821438342    AK  1821438342                2        99559          both   
2   1225101041    AK  1225101041                2        99762          both   
3   1952482036    AK  1952482036                2        71742          both   
4   1598763740    AK  1598763740                2        99835          both   
5   1609843523    AK  1609843523                2        84604          both   
6   1447380423    AK  1447380423                2        72703          both   
7   1275508558    AK  1275508558                2        66071          both   
8   1679566269    AK  1679566269                2        99603          both   
9   1528062429    AK  1528062429                2        99669          both   

     ZIP COUNTY_SSA      County_name STATE_CD STATE_NAME  \
0  99835        220    Sitka Borough       02     Alas

In [30]:
facility_area_wage_index = pd.concat([urban_cbsa_wage_index_merge, rural_cbsa_wage_index_merge])
facility_area_wage_index = facility_area_wage_index.sort_values(by=['STATE'])

In [31]:
print(facility_area_wage_index.shape[0])

19412


In [32]:
facility_area_wage_index['Wage_Index'] = facility_area_wage_index['Wage_Index'].replace('-----', np.nan)
facility_area_wage_index['Wage_Index'] = facility_area_wage_index['Wage_Index'].astype(float)

In [18]:
print(facility_are_wage_index

BLG_PRVDR_NPI                     object
STATE                             object
NPI                       string[python]
Entity_Type_Code          string[python]
Practice_Zip                      object
taf_npi_merge                   category
ZIP                               object
COUNTY_SSA                        object
County_name                       object
STATE_CD                          object
STATE_NAME                        object
taf_provider_zip_merge          category
county                            object
state                             object
ssacd                             object
fipscounty                        object
cbsa                              object
cbsaname                          object
county_ssa_merge                category
CBSA_Code                         object
Wage_Index                       float64
urban_CBSA_merge                category
urban/rural                       object
State Code                        object
Nonurban Area   

In [13]:

facility_area_wage_index['facility_npi'] = facility_area_wage_index['facility_npi'].astype('str')

# read out to csv 
facility_area_wage_index.to_csv('/gpfs/data/cms-share/duas/56930/Nadia/python_code/medicaid_pmt_rates/wage_index/2019_facility_area_wage_index.csv')




   Unnamed: 0.1  Unnamed: 0 facility_npi STATE           NPI  \
0             0           1   1801021290    AK  1.801021e+09   
1             3          37   1629198403    AK  1.629198e+09   
2             5          39   1376606186    AK  1.376606e+09   
3             6          40   1275660128    AK  1.275660e+09   
4             7          41   1225101041    AK  1.225101e+09   
5             9          43   1942290267    AK  1.942290e+09   
6            10          44   1730548462    AK  1.730548e+09   
7            11          45   1891804308    AK  1.891804e+09   
8            12          46   1306981238    AK  1.306981e+09   
9            15          49   1770610438    AK  1.770610e+09   

   Entity_Type_Code  Practice_Zip taf_npi_merge      ZIP  COUNTY_SSA  \
0               2.0       99504.0          both  99504.0        20.0   
1               2.0       99762.0          both  99762.0       180.0   
2               2.0       87120.0          both  87120.0         0.0   
3      